# Lecture 3 — Policies, Timing, Populations

**Computational Methods for Heterogeneous-Agent Macro**

Jeffrey Sun


### Environment


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra

# Core Code


### Parameters


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
function get_params(;
        β=0.96,
        R=1.04,
        b_grid=loggrid(0.05, 200.0, 400),
        z_grid=[2.7, 2.6, 2.5],
        P_z=[0.95 0.04 0.01;
           0.05 0.9  0.05;
           0.01 0.04 0.95],
    )

    return params = (;β, R,
                    V_shape=(length(b_grid), length(z_grid)),
                    b_grid, z_grid, P_z)
end

In [ ]:
params = get_params(; β=0.90)

### Markov Income

Human capital $z$ takes three values (high, medium, low) — here $z \in \{2.7, 2.6, 2.5\}$ — with a row-stochastic transition matrix $\Pi$ that places most of its mass on the diagonal: spells of each state are persistent.


### Utility and Log Grid


### Simple Discretized Grid Functions
These functions are as simple as possible for clarity. In reality, they leave a lot to be desired: interpolation, better performance, etc.


In [ ]:
"""
Snap a value `x` to the nearest grid index.

"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

In [ ]:
closest_b_ind = snap_idx(params.b_grid, 186.05)

params.b_grid[closest_b_ind]

### Backward iteration in stages

The three stages compose into one backward step on a 2-D value function
$V \in \mathbb{R}^{N_b \times N_z}$. Each stage maps one labelled $V$ to the next:

- **Stage 3 (Consumption-Saving) backward.** Grid-search $V^{\mathrm{end}} \mapsto V$.
- **Stage 2 (Income) backward.** Lookup $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$ at $R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j]$.
- **Stage 1 (Income Shock) backward.** Matrix-multiply $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$ by $\Pi^\top$.

At the period boundary, $V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$.


In [ ]:
"""
Stage 1 (Income Shock) backward: V_start = V_pre_inc * P_z'.

"""
income_shock_backward(V_post_inc_shock, params) = V_post_inc_shock * params.P_z'

"""
Stage 2 (Income) backward: V_pre_inc(b_end, z) = V(R*b_end + z, z), snapped to nearest grid.
Fully broadcast — no explicit loop.

"""
function income_backward(V_post_inc, params)
    (;R, b_grid, z_grid) = params
    b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')
    return V_pre_inc = V_post_inc[CartesianIndex.(b_post_inc, axes(b_post_inc, 2)')]
end


In [ ]:
params = get_params()

(;R, b_grid, z_grid) = params
b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')

# Each entry of b_post_inc is end-of-stage-2 wealth (b) index


In [ ]:
"""
Stage 3 (Consumption-Saving) backward: grid-search V_end → V over b_end choices.

"""
function consumption_saving_backward(V_end, params)
    # Unpack b_grid
    (;b_grid) = params

    # Move b_grid to dimension 3 to represent b_end choices
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    V = maximum(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    # Drop dimension 3 from V and return
    return dropdims(V; dims=3)
end

In [ ]:
V_end = zeros(params.V_shape)
@show size(V_end)

(;b_grid) = params #b_grid = params.b_grid

b_next_grid = insertdims(b_grid; dims=(1,2))
@show size(b_next_grid)

@show size(V_end)
V_end_reshape = permutedims(insertdims(V_end; dims=3), (3,2,1))
@show size(V_end_reshape)

objective = u.(b_grid .- b_next_grid) .+ V_end_reshape
@show size(objective)

V_pre_consume = maximum(objective; dims=3)
@show size(V_pre_consume)

dropdims(V_pre_consume; dims=3)


In [ ]:
function bellman_operator(V_end, params)
    (;β) = params
    # Stage 3
    V_post_inc = consumption_saving_backward(V_end, params)

    # Stage 2
    V_post_inc_shock = income_backward(V_post_inc, params)

    # Stage 1
    V_start = income_shock_backward(V_post_inc_shock, params)

    # Passage of time
    V_end_new = β .* V_start
    return V_end_new
end

In [ ]:
params = get_params()
V_end = zeros(params.V_shape)
V_end_new = bellman_operator(V_end, params)

### Value Function Iteration (Same as L2!)

Iterate $V \mapsto \mathcal{T}V$ until the max-abs change drops below `tol`.


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    V = zeros(params.V_shape)

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")
    return V
end

In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)

### A Population is an Array (Vector, Matrix, etc.)

A *population* can be thought of as a *distribution* over state variables.

If $b$ is the only state variable, then a *population* is just a vector $\lambda$, with $\lambda_i$ = "number of people with wealth $b_i$."

If $(b, z)$ are the state variables, the population is a matrix $\lambda$, with $\lambda_{ij}$ = "number of people with wealth $b_i$ and human capital $z_j$."


### Forward iteration in stages

We can simulate each population matrix forward though each stage, one at a time.

- **Stage 1 (Income Shock) forward.** $\lambda \mapsto \lambda\, \Pi$.
- **Stage 2 (Income) forward.** Each $(b^{\mathrm{end}}, z)$ cell moves to $(R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j],\, z)$.
- **Stage 3 (Consumption-Saving) forward.** Each $(b, z)$ cell moves to $(b - c^\star(b, z),\, z)$.

Both wealth re-bins snap to the nearest grid point.


In [ ]:
"""
Stage 1 forward: λ_post = λ_pre * P_z.

"""
income_shock_forward(λ, P_z) = λ * P_z

"""
Stage 2 forward: each (i_b_end, i_z) cell moves to (snap(R*b_end + z), i_z).
Vectorize the destination index via broadcasting; one tight scatter.

"""

function change_λ_wealth(λ, b_inds_new)
    λ_new = zero(λ)
    for old_idx in CartesianIndices(λ)
        λ_new[b_inds_new[old_idx], old_idx[2]] += λ[old_idx]
    end
    return λ_new
end

function income_forward(λ, params)
    (;R, b_grid, z_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')               # (N_b, N_z)
    return change_λ_wealth(λ, dest)
end

"""
Stage 3 forward: each (i_b, i_z) cell moves to (c_ind[i_b, i_z], i_z).
c_ind is exactly the destination index — direct scatter.

"""
function policy_function(V_end, params)
    # Unpack b_grid
    (;b_grid) = params

    # Move b_grid to dimension 3 to represent b_end choices
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    policy_idxs = argmax(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    return [idx[3] for idx in policy_idxs[:,:,1]]
end

consumption_saving_forward(λ, b_inds_new, params) = change_λ_wealth(λ, b_inds_new)

"""
One forward step of the composite operator T*:
λ → income shock → income → consumption saving.

"""
function simulate_population_forward(λ, b_inds_new, params)
    # Stage 1
    λ = income_shock_forward(λ, params.P_z)

    # Stage 2
    λ = income_forward(λ, params)

    # Stage 3
    λ = consumption_saving_forward(λ, b_inds_new, params)

    return λ
end


In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

# Guess λ
λ_start = fill(1/length(params.V_shape), params.V_shape)
@show sum(λ_start)


In [ ]:
function find_steady_state_population(V_end, params; tol=1e-5, maxiter=10_000, verbosity=0)

    b_inds_new = policy_function(V_end, params)

    λ = fill(1/length(params.V_shape), params.V_shape) # Start with households evenly distributed across gridpoints

    Δ, iters = Inf, 0
    while Δ >= tol
        λ_new = simulate_population_forward(λ, b_inds_new, params)
        Δ = maximum(abs.(λ_new .- λ))
        λ = λ_new
        iters += 1
        iters > maxiter && error("Steady state λ did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("Steady state λ converged in $iters iterations with error $Δ")
    return λ
end

In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
λ_steady_state = find_steady_state_population(V_end, params; verbosity=1)

In [ ]:
plot(vec(λ_steady_state[:,1]))

## Demo

Set up parameters, solve the value function, read off the policy, and run both simulators.


In [ ]:
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

λ_start = fill(1/length(params.V_shape), params.V_shape)
λ_next = simulate_population_forward(λ_start, b_inds_new, params)
;

### Plot $V$ and $c^\star$


In [ ]:
(;b_grid, z_grid) = params
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)

# turn the index vector b_inds_new back into a consumption value for plotting:
c_pol = b_grid .- b_grid[b_inds_new]

p1 = plot(b_grid, V_end; lw=2, xlabel="wealth b", ylabel="V(b)",
          label=["high z" "medium z" "low z"], legend=:bottomright)
p2 = plot(b_grid, c_pol; lw=2, xlabel="wealth b", ylabel="c*(b)",
          label="c*", legend=:bottomright)
plot!(p2, b_grid, b_grid; ls=:dash, lw=1, label="45°")
plot(p1, p2; layout=(1, 2), size=(1000, 320))


### Simulate one household

Roll a single household forward over time. Wealth drifts up during high-$z$ spells and drops when human capital switches to a lower state.


In [ ]:
"""
Draw the next state of a Markov chain with row-stochastic P_z, given current state i.

"""
sample_markov(P_z, i) = findfirst(rand() .<= cumsum(P_z[i, :]))

"""
Roll one household forward T periods using the policy index matrix `c_inds`.
Each period applies Stage 1 (Markov on z), Stage 2 (cash-on-hand
b = R·b_end + z), Stage 3 (savings = policy lookup at (b, z)). The recorded
`i_b_path[t]` is cash-on-hand at t (wealth after z realizes).

"""
function simulate_one(c_inds, params; T=200, b_ind_init=1, z_ind_init=1)
    (;R, z_grid, P_z, b_grid) = params

    # State carried across iterations: end-of-period savings index `b_ind`
    # (b_end) and current human-capital state `z_ind`. A fresh income shock is
    # drawn first each period, matching the timing of the backward Bellman.
    b_ind, z_ind = b_ind_init, z_ind_init

    i_b_path, i_z_path = zeros(Int, T), zeros(Int, T)
    for t in 1:T
        # Stage 1: income shock — draw next z state.
        z_ind = sample_markov(P_z, z_ind)

        # Stage 2: realize income, assemble cash-on-hand b = R·b_end + z.
        b_cash_ind = snap_idx(b_grid, R * b_grid[b_ind] + z_grid[z_ind])

        # Stage 3: savings = policy at (cash-on-hand, z).
        b_ind = c_inds[b_cash_ind, z_ind]

        i_b_path[t], i_z_path[t] = b_cash_ind, z_ind
    end
    return (;i_b_path, i_z_path)
end

In [ ]:
(;b_grid, z_grid) = params
params = get_params()
V_end = solve_vfi(params; verbosity=1)
b_inds_new = policy_function(V_end, params)


Random.seed!(1)
# start at the grid point nearest to wealth 1.0, high z
sim = simulate_one(b_inds_new, params; T=300,
                   b_ind_init=argmin(abs.(b_grid .- 1.0)), z_ind_init=1)

t_axis = 1:length(sim.i_b_path)
p1 = plot(t_axis, b_grid[sim.i_b_path]; lw=1.5,
          xlabel="t", ylabel="wealth b_t",
          label="b_t", legend=:topright)
p2 = plot(t_axis, [z_grid[z] for z in sim.i_z_path]; lw=1.5,
          xlabel="t", ylabel="human capital z_t",
          label="z_t", legend=:topright, seriestype=:steppost)
plot(p1, p2; layout=(2, 1), size=(820, 420))


In [ ]:
# start everyone at wealth ≈ 1.0, high z
λ_0 = zeros(length(b_grid), length(z_grid))
i_0 = argmin(abs.(b_grid .- 1.0))
λ_0[i_0, 1] = 1.0

λ_path = simulate_population(c_ind, params, λ_0; T=200)
;


In [ ]:
# marginal wealth distribution at four snapshots
function wealth_marginal(λ)
    return vec(sum(λ; dims=2))
end

snapshots = [1, 5, 25, 200]
p = plot(xlabel="wealth b", ylabel="density", legend=:topright)
for t in snapshots
    plot!(p, b_grid, wealth_marginal(λ_path[t]); lw=1.8, label="t = $(t-1)")
end
plot(p; size=(820, 320), xscale=:log10)


### Cross-sectional aggregates from $V$ and $c^\star$

Aggregate welfare $\bar V$, consumption $\bar C$, wealth $\bar K$ at each $t$ — all inner products against the marginal wealth distribution.


In [ ]:
# inner products against the marginal wealth distribution
V_bar = [dot(V,      wealth_marginal(λ)) for λ in λ_path]
C_bar = [dot(c_pol,  wealth_marginal(λ)) for λ in λ_path]
K_bar = [dot(b_grid, wealth_marginal(λ)) for λ in λ_path]

t_axis = 0:length(λ_path)-1
p1 = plot(t_axis, V_bar; lw=1.8, xlabel="t", ylabel="V̄_t", label="V̄")
p2 = plot(t_axis, C_bar; lw=1.8, xlabel="t", ylabel="C̄_t", label="C̄")
p3 = plot(t_axis, K_bar; lw=1.8, xlabel="t", ylabel="K̄_t", label="K̄")
plot(p1, p2, p3; layout=(1, 3), size=(1000, 280))


# Step-by-Step Explanation


### Policy via `argmax`, broken down


In [ ]:
# Re-do L02's deterministic Bellman (constant z = 1.0) on a 1-D V slice,
# to expose what argmax returns on the (b, b') matrix.
z_const = 1.0
b_end = (b_grid' .- z_const) ./ params.R
M     = u.(b_grid .- b_end) .+ params.β .* V'
size(M)


In [ ]:
# `argmax` along columns: for each row (each b), which column (b') is best?
idx = argmax(M; dims=2)
idx[1:5]      # CartesianIndex(i, j*) for the first five b


In [ ]:
# pull out just the column index — this is the chosen b'-index
j_star = vec(getindex.(idx, 2))
j_star[1:5]


In [ ]:
# the chosen b'-index for each b is exactly what policy_function returns
c_ind_check = vec(getindex.(idx, 2))
maximum(abs.(c_ind_check .- c_ind))   # should be 0


### Markov along one dimension is matrix multiplication

Take just the $z$ marginal $\lambda_z$ (length 3). One Markov step is $\lambda_z \cdot \Pi$. After many steps the $z$ shares stop changing — this is the long-run share of each human-capital state implied by $\Pi$.


In [ ]:
# start: everyone in the high-z state
λ_z_0 = [1.0  0.0  0.0]                         # row vector

# one step
λ_z_1 = λ_z_0 * params.P_z

# iterate forward many times; the z shares settle to their long-run values
λ_z = let λ = λ_z_0
    for t in 1:200
        λ = λ * params.P_z
    end
    λ
end

(long_run_high_z=λ_z[1], long_run_medium_z=λ_z[2], long_run_low_z=λ_z[3])


### Trace one household through the three stages

Pick a single $(i_b, i_z)$ cell, draw a Markov step, then apply Stages 2--3 (the cash-on-hand assembly + index lookup) to see where its mass ends up.


In [ ]:
# trace one (i_b, i_z) cell through Stages 1, 2, 3
i_b, i_z = 50, 1

# Stage 1: a Markov draw advances the z state
Random.seed!(42)
i_z_new = sample_markov(params.P_z, i_z)

# Stage 2: assemble cash-on-hand b = R·b_end + z using the realized z, snap to grid
b_cash    = params.R * b_grid[i_b] + z_grid[i_z_new]
i_b_cash  = snap_idx(b_grid, b_cash)

# Stage 3: savings = policy lookup at (cash-on-hand, z)
i_b_new = b_inds_new[i_b_cash, i_z_new]
b_new   = b_grid[i_b_new]

(i_b=i_b, i_z=i_z, i_z_new=i_z_new,
 b_cash=b_cash, i_b_cash=i_b_cash,
 i_b_new=i_b_new, b_new=b_new)

Iterate the forward step many times. The wealth distribution settles into a long-run shape; aggregates read off as inner products with $V$, $c^\star$, and $b$.


In [ ]:
# start everyone at the same wealth and z, then apply the forward step
# many times — the distribution stops changing once we've iterated long enough.
λ_long_run = let λ = zeros(length(params.b_grid), length(params.z_grid))
    i_0 = argmin(abs.(params.b_grid .- 1.0))
    λ[i_0, 1] = 1.0
    for t in 1:400
        λ = forward_step(λ, c_ind_2d, params)
    end
    λ
end

# aggregates in the long run
λ_z = vec(sum(λ_long_run; dims=1))         # z marginal
λ_b = vec(sum(λ_long_run; dims=2))         # wealth marginal

(high_z_share=λ_z[1],
 mean_wealth=dot(params.b_grid, λ_b),
 mean_V=sum(V_2d .* λ_long_run))
